# Week11 예습과제 — 6장 Word2Vec ~ 순환신경망


## Word2Vec · Skip-gram (교재 p.286~298)

In [7]:
# 추가 셀
from torch import nn

In [8]:
# 추가 셀

!pip install Korpora konlpy


In [9]:
# p.286 | 임베딩 계층 기본 형태

embedding = nn.Embedding(
    num_embeddings=5000,
    embedding_dim=128,
    padding_idx=None,
    max_norm=None,
    norm_type=2.0
)


In [10]:
# p.287 | 예제 6.3 | 기본 Skip-gram 클래스

from torch import nn

class VanillaSkipgram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim
        )
        self.linear = nn.Linear(
            in_features=embedding_dim,
            out_features=vocab_size
        )

    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        output = self.linear(embeddings)
        return output


In [11]:
# p.288 | 예제 6.4 | 영화 리뷰 데이터셋 전처리

import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)

tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])



    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ra

In [12]:
# p.288 | 예제 6.5 | 단어 사전 구축

from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

vocab = build_vocab(corpus=tokens, n_vocab=5000, special_tokens=["<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}

id_to_token = {idx: token for idx, token in enumerate(vocab)}
print(vocab[:10])
print(len(vocab))


['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [13]:
# p.289 | 예제 6.6 | Skip-gram 단어 쌍 추출 함수

def get_word_pairs(tokens, window_size):
    pairs = []
    for sentence in tokens:
        sentence_length = len(sentence)
        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx - window_size)
            window_end = min(sentence_length, idx + window_size + 1)
            center_word = sentence[idx]
            context_words = sentence[window_start:idx] + sentence[idx + 1:window_end]
            for context_word in context_words:
                pairs.append([center_word, context_word])
    return pairs

word_pairs = get_word_pairs(tokens, window_size=2)
print(word_pairs[:5])


[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


In [14]:
# p.290 | 예제 6.7 | 단어 쌍을 인덱스 쌍으로 변환

def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id["<unk>"]
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append((center_index, context_index))
    return pairs

index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])


[(595, 100), (100, 595), (77, 176), (77, 2), (176, 77)]


In [15]:
# p.291 | 예제 6.8 | 데이터로더 적용

import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs = torch.tensor(index_pairs)
center_indexes = index_pairs[:, 0]
context_indexes = index_pairs[:, 1]

dataset = TensorDataset(center_indexes, context_indexes)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)


In [16]:
# p.291 | 예제 6.9 | Skip-gram 모델 준비 작업

from torch import optim

device = "cuda" if torch.cuda.is_available() else "cpu"
word2vec = VanillaSkipgram(vocab_size=len(token_to_id), embedding_dim=128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr=0.1)


In [ ]:
# p.292 | 예제 6.10 | Skip-gram 모델 학습

for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss

    cost = cost / len(dataloader)
    print(f"Epoch : {epoch + 1}, Cost : {cost:.3f}")


In [ ]:
# p.293 | 예제 6.11 | 단어 임베딩 추출

token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()

for word, token in token_to_id.items():
    token_to_embedding[word] = embedding_matrix[token]

index = 30
token = vocab[30]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)


In [ ]:
# p.294 | 예제 6.12 | 단어 임베딩 유사도 계산

import numpy as np
from numpy.linalg import norm

def cosine_similarity(a, b):
    cosine = np.dot(a, b) / (norm(a, axis=-1) * norm(b))
    return cosine

def top_n_index(cosine_matrix, n):
    closest_indexes = cosine_matrix.argsort()[::-1]
    top_n = closest_indexes[1:n + 1]
    return top_n

cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n=5)

print(f"{token}와 가장 유사한 5개 단어")
for index in top_n:
    print(f"{id_to_token[index]} - 유사도 : {cosine_matrix[index]:.4f}")


In [ ]:
# p.295 | 젠심 라이브러리 설치
!pip install gensim


In [ ]:
# 추가 셀

from gensim.models import Word2Vec

In [ ]:
# p.296 | Word2Vec 클래스

word2vec = Word2Vec(
    sentences=tokens,
    corpus_file=None,
    vector_size=100,
    alpha=0.025,
    window=5,
    min_count=5,
    workers=3,
    sg=0,
    hs=0,
    cbow_mean=1,
    negative=5,
    ns_exponent=0.75,
    max_final_vocab=None,
    epochs=5,
    batch_words=10000
)


In [ ]:
# p.297 | 예제 6.13 | Gensim Word2Vec 모델 학습

from gensim.models import Word2Vec

word2vec = Word2Vec(
    sentences=tokens,
    vector_size=128,
    window=5,
    min_count=1,
    sg=1,
    epochs=3,
    max_final_vocab=10000
)

# word2vec.save("./models/word2vec.model")
# word2vec = Word2Vec.load("./models/word2vec.model")


In [ ]:
# p.298 | 예제 6.14 | 임베딩 추출 및 유사도 계산

word = "연기"
print(word2vec.wv[word])
print(word2vec.wv.most_similar(word, topn=5))
print(word2vec.wv.similarity(word, w2="연기력"))


## fastText (교재 p.301~304)

In [ ]:
# 추가 셀

import gensim

In [ ]:
# p.301 | fastText 클래스


fasttext = gensim.models.FastText(
    sentences=tokens,
    corpus_file=None,
    vector_size=100,
    alpha=0.025,
    window=5,
    min_count=5,
    workers=3,
    sg=0,
    hs=0,
    cbow_mean=1,
    negative=5,
    ns_exponent=0.75,
    max_final_vocab=None,
    epochs=5,
    batch_words=10000,
    min_n=3,
    max_n=6
)


In [ ]:
# p.302 | 예제 6.15 | KorNLI 데이터세트 전처리

from Korpora import Korpora

corpus = Korpora.load("kornli")
corpus_texts = corpus.get_all_texts() + corpus.get_all_pairs()
tokens = [sentence.split() for sentence in corpus_texts]

print(tokens[:3])


In [ ]:
# p.303 | 예제 6.16 | fastText 모델 실습

from gensim.models import FastText

fastText = FastText(
    sentences=tokens,
    vector_size=128,
    window=5,
    min_count=5,
    sg=1,
    epochs=3,
    min_n=2,
    max_n=6
)

# fastText.save("./models/fastText.model")
# fastText = FastText.load("./models/fastText.model")


In [ ]:
# p.304 | 예제 6.17 | fastText OOV 처리

oov_token = "사랑해요"
oov_vector = fastText.wv[oov_token]
print(oov_token in fastText.wv.key_to_index)
print(fastText.wv.most_similar(oov_vector, topn=5))


## 순환 신경망 · RNN (교재 p.311~313)

In [ ]:
# 추가 셀
from torch import nn

In [ ]:
# p.311 | 순환 신경망 클래스

rnn =torch.nn.RNN(
    input_size=128,
    hidden_size=256,
    num_layers=3,
    nonlinearity="tanh",
    bias=False,
    batch_first=True,
    dropout=0.0,
    bidirectional=False
)


In [ ]:
# p.312~313 | 예제 6.18 | 양방향 다층 신경

import torch
from torch import nn

input_size = 128
output_size = 256
num_layers = 3
bidirectional = True

model = nn.RNN(
    input_size=input_size,
    hidden_size=output_size,
    num_layers=num_layers,
    nonlinearity="tanh",
    batch_first=True,
    bidirectional=bidirectional
)

batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.randn(num_layers * (int(bidirectional) + 1), batch_size, output_size)

outputs, hidden = model(inputs, h_0)
print(outputs.shape)
print(hidden.shape)


## LSTM (교재 p.320~335)

In [ ]:
# p.320 | 장단기 메모리 클래스

lstm = torch.nn.LSTM(
    input_size=128,
    hidden_size=256,
    num_layers=3,
    bias=False,
    batch_first=True,
    dropout=0.,
    bidirectional=False,
    proj_size=0
)


In [ ]:
# p.321~322 | 예제 6.19 | 양방향 다층 장단기 메모리

import torch
from torch import nn

input_size = 128
output_size = 256
num_layers = 3
bidirectional = True
proj_size = 64

model = nn.LSTM(
    input_size=input_size,
    hidden_size=output_size,
    num_layers=num_layers,
    batch_first=True,
    bidirectional=bidirectional,
    proj_size=proj_size
)

batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.randn(num_layers * (int(bidirectional) + 1), batch_size, proj_size if proj_size > 0 else output_size)
c_0 = torch.randn(num_layers * (int(bidirectional) + 1), batch_size, output_size)

outputs, (h_n, c_n) = model(inputs, (h_0, c_0))
print(outputs.shape)
print(h_n.shape)
print(c_n.shape)


In [ ]:
# p.324~325 | 예제 6.20 | 문장 분류 모델

class SentenceClassifier(nn.Module):
    def __init__(
        self,
        n_vocab,
        hidden_dim,
        embedding_dim,
        n_layers,
        dropout=0.5,
        bidirectional=True,
        model_type="lstm"
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=n_vocab,
            embedding_dim=embedding_dim,
            padding_idx=0
        )
        if model_type == "rnn":
            self.model = nn.RNN(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                nonlinearity="tanh",
                bidirectional=bidirectional,
                batch_first=True,
                dropout=dropout
            )
        elif model_type == "lstm":
            self.model = nn.LSTM(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                bidirectional=bidirectional,
                dropout=dropout,
                batch_first=True
            )
        if bidirectional:
            self.classifier = nn.Linear(hidden_dim * 2, 1)
        else:
            self.classifier = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        outputs, _ = self.model(embeddings)
        last_output = outputs[:, -1, :]
        last_output = self.dropout(last_output)
        logits = self.classifier(last_output)
        return logits


In [ ]:
# p.325~326 | 예제 6.21 | 데이터세트 불러오기

import pandas as pd
from Korpora import Korpora

corpus = Korpora.load("nsmc")
corpus_df = pd.DataFrame(corpus.test)

train = corpus_df.sample(frac=0.9, random_state=42)
test = corpus_df.drop(train.index)

print(train.head(5).to_markdown())
print("Training Data Size :", len(train))
print("Testing Data Size :", len(test))


In [ ]:
# p.326~327 | 예제 6.22 | NSMC 토큰화 및 어휘 사전 구축

from konlpy.tag import Okt
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

tokenizer = Okt()
train_tokens = [tokenizer.morphs(review) for review in train.text]
test_tokens = [tokenizer.morphs(review) for review in test.text]

vocab = build_vocab(corpus=train_tokens, n_vocab=5000, special_tokens=["<pad>", "<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))


In [ ]:
# p.327~328 | 예제 6.23 | 정수 인코딩 및 패딩

import numpy as np

def pad_sequences(sequences, max_length, pad_value):
    result = list()
    for sequence in sequences:
        sequence = sequence[:max_length]
        pad_length = max_length - len(sequence)
        padded_sequence = sequence + [pad_value] * pad_length
        result.append(padded_sequence)
    return np.asarray(result)

unk_id = token_to_id["<unk>"]
train_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in train_tokens
]
test_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in test_tokens
]

max_length = 32
pad_id = token_to_id["<pad>"]
train_ids = pad_sequences(train_ids, max_length, pad_id)
test_ids = pad_sequences(test_ids, max_length, pad_id)

print(train_ids[0])
print(test_ids[0])


In [ ]:
# p.329 | 예제 6.24 | 데이터로더 적용

import torch
from torch.utils.data import TensorDataset, DataLoader

train_ids = torch.tensor(train_ids)
test_ids = torch.tensor(test_ids)

train_labels = torch.tensor(train.label.values, dtype=torch.float32)
test_labels = torch.tensor(test.label.values, dtype=torch.float32)

train_dataset = TensorDataset(train_ids, train_labels)
test_dataset = TensorDataset(test_ids, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [ ]:
# p.329~330 | 예제 6.25 | 손실 함수와 최적화 함수 정의

from torch import optim

n_vocab = len(token_to_id)
hidden_dim = 64
embedding_dim = 128
n_layers = 2

device = "cuda" if torch.cuda.is_available() else "cpu"
classifier = SentenceClassifier(
    n_vocab=n_vocab, hidden_dim=hidden_dim, embedding_dim=embedding_dim, n_layers=n_layers
).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr=0.001)


In [ ]:
# p.330~332 | 예제 6.26 | 훈련 및 테스트 함수

def train(model, datasets, criterion, optimizer, device, interval):
    model.train()
    losses = list()

    for step, (input_ids, labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % interval == 0:
            print(f"Train Loss {step} : {np.mean(losses)}")


def test(model, datasets, criterion, device):
    model.eval()
    losses = list()
    corrects = list()

    with torch.no_grad():
        for step, (input_ids, labels) in enumerate(datasets):
            input_ids = input_ids.to(device)
            labels = labels.to(device).unsqueeze(1)

            logits = model(input_ids)
            loss = criterion(logits, labels)
            losses.append(loss.item())
            yhat = torch.sigmoid(logits) > .5
            corrects.extend((yhat == labels).cpu().numpy().tolist())

    print(f"Val Loss : {np.mean(losses)}, Val Accuracy : {np.mean(corrects)}")

epochs = 5
interval = 500

for epoch in range(epochs):
    train(classifier, train_loader, criterion, optimizer, device, interval)
    test(classifier, test_loader, criterion, device)


In [ ]:
# p.333 | 예제 6.27 | 학습된 모델로부터 임베딩 추출

token_to_embedding = dict()
embedding_matrix = classifier.embedding.weight.detach().cpu().numpy()

for word, token in token_to_id.items():
    token_to_embedding[word] = embedding_matrix[token]

token = vocab[1000]
print(token, token_to_embedding[token])


In [ ]:
# p.333 | 예제 6.28 | 사전 학습된 모델로 임베딩 계층 초기화

from gensim.models import Word2Vec

word2vec = Word2Vec.load("./models/word2vec.model")
init_embeddings = np.zeros((n_vocab, embedding_dim))

for index, token in id_to_token.items():
    if token not in ["<pad>", "<unk>"]:
        init_embeddings[index] = word2vec.wv[token]

embedding_layer = nn.Embedding.from_pretrained(
    torch.tensor(init_embeddings, dtype=torch.float32)
)


In [ ]:
# 추가 셀
from torch import nn
import torch

In [ ]:
# p.334 | 예제 6.29 | 사전 학습 임베딩 계층 적용

class SentenceClassifier(nn.Module):
    def __init__(self, n_vocab, hidden_dim, embedding_dim, n_layers,
                 dropout=0.5, bidirectional=True, model_type="lstm",
                 pretrained_embedding=None):
        super().__init__()

        if pretrained_embedding is not None:
            self.embedding = nn.Embedding.from_pretrained(
                torch.tensor(pretrained_embedding, dtype=torch.float32)
            )
        else:
            self.embedding = nn.Embedding(n_vocab, embedding_dim, padding_idx=0)
        if model_type == "rnn":
            self.model = nn.RNN(embedding_dim, hidden_dim, n_layers,
                                nonlinearity="tanh", bidirectional=bidirectional,
                                batch_first=True, dropout=dropout)
        else:
            self.model = nn.LSTM(embedding_dim, hidden_dim, n_layers,
                                 bidirectional=bidirectional, dropout=dropout,
                                 batch_first=True)
        self.classifier = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        outputs, _ = self.model(embeddings)
        last_output = outputs[:, -1, :]
        last_output = self.dropout(last_output)
        return self.classifier(last_output)


In [ ]:
# p.335 | 예제 6.30 | 사전 학습된 임베딩을 사용한 모델 학습

classifier = SentenceClassifier(
    n_vocab=n_vocab, hidden_dim=hidden_dim,
    embedding_dim=embedding_dim, n_layers=n_layers,
    pretrained_embedding=init_embeddings
).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr=0.001)

epochs = 5
interval = 500
for epoch in range(epochs):
    train(classifier, train_loader, criterion, optimizer, device, interval)
    test(classifier, test_loader, criterion, device)